# Tampered Image Detection and Localization — Submission Notebook (vK.4)

This **Kaggle** notebook presents a complete assignment submission for tampered image detection and tampered region localization.

**Key improvements in vK.4 over vK.2**
- Kaggle-native: no Colab/Drive shims — dataset mounts automatically
- Centralized `CONFIG` dict for all hyperparameters
- Full reproducibility via `set_seed()`
- Mixed-precision training (AMP) + gradient accumulation
- `pos_weight` BCE + per-sample Dice loss (from v8)
- Expanded augmentation pipeline (ColorJitter, Compression, GaussNoise, GaussianBlur)
- ReduceLROnPlateau scheduler + early stopping
- Threshold sweep, mask-size stratified evaluation
- Grad-CAM explainability, robustness testing, shortcut learning checks

**Model architecture preserved from vK.2:** custom `UNetWithClassifier` (DoubleConv encoder-decoder + classification head).

## Project Objectives

| Status | Objective | Notes |
|---|---|---|
| ✅ | Detect whether an image is tampered | Classifier head → image-level accuracy |
| ✅ | Localize tampered regions | Segmentation branch → pixel-level masks |
| ✅ | Run in one Kaggle notebook | Single notebook, GPU P100 |
| ✅ | Track experiments | W&B integration (optional) |
| ✅ | Explainability | Grad-CAM, failure analysis, robustness testing |

## 1. Environment Setup

Install dependencies and configure the central `CONFIG` dictionary.

In [ ]:
!pip install -q "albumentations>=1.3.1,<2.0" opencv-python-headless

import os
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # set before importing albumentations

import sys, random, json, warnings, math, contextlib
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Suppress C-level libpng warnings
_devnull = os.open(os.devnull, os.O_WRONLY)
os.dup2(_devnull, 2)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / (1024**3):.1f} GB")

In [ ]:
# ── Central Configuration ─────────────────────────────────────────────────────

SEED = 42

CONFIG = {
    # ── Data ──
    'image_size': 256,
    'batch_size': 16,
    'num_workers': 4,

    # ── Model ── (custom UNetWithClassifier from vK.2)
    'n_channels': 3,
    'n_classes': 1,      # segmentation output channels
    'n_labels': 2,       # classification labels (authentic / tampered)

    # ── Optimizer ──
    'lr': 1e-4,
    'weight_decay': 1e-4,

    # ── Scheduler ──
    'scheduler': 'reduce_on_plateau',
    'scheduler_patience': 3,
    'scheduler_factor': 0.5,
    'scheduler_min_lr': 1e-6,

    # ── Training ──
    'max_epochs': 50,
    'patience': 10,           # early stopping patience
    'accumulation_steps': 2,  # effective batch = 16 * 2 = 32
    'max_grad_norm': 1.0,

    # ── Loss ──
    'use_pos_weight': True,
    'dice_per_sample': True,
    'cls_loss_weight': 1.5,   # ALPHA
    'seg_loss_weight': 1.0,   # BETA

    # ── Augmentation ──
    'aug_color_jitter': True,
    'aug_compression': True,
    'aug_gauss_noise': True,
    'aug_gauss_blur': True,

    # ── Feature Flags ──
    'use_amp': True,
    'use_wandb': True,

    # ── Reproducibility ──
    'seed': SEED,
}

# ── Output directories ──
OUTPUT_DIR = '/kaggle/working'
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')

for d in [CHECKPOINT_DIR, RESULTS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print('CONFIG:')
for k, v in CONFIG.items():
    print(f'  {k}: {v}')
print(f'\nEffective batch size: {CONFIG["batch_size"] * CONFIG["accumulation_steps"]}')

In [ ]:
# ── Device & Seed ─────────────────────────────────────────────────────────────

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    if hasattr(torch.backends, 'cuda') and hasattr(torch.backends.cuda, 'matmul'):
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.allow_tf32 = True
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')

print(f'Training device: {device}')
print(f'Seed: {SEED}')

## 2. Experiment Tracking (W&B)

Kaggle-native W&B setup using `kaggle_secrets`.

In [ ]:
WANDB_ACTIVE = False
WANDB_RUN = None

if CONFIG['use_wandb']:
    try:
        !pip install -q wandb
        import wandb
        from kaggle_secrets import UserSecretsClient
        wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
        WANDB_RUN = wandb.init(
            project='tampered-image-detection-assignment',
            config=CONFIG,
            name=f'vk4-unetwithclassifier-kaggle-seed{SEED}',
            tags=['vK.4', 'casia-v2', 'kaggle', 'custom-unet'],
        )
        WANDB_ACTIVE = True
        print('W&B initialized.')
    except Exception as e:
        print(f'W&B setup failed, continuing without: {e}')
        WANDB_ACTIVE = False
else:
    print('W&B disabled.')

print(f'W&B active: {WANDB_ACTIVE}')

## 3. Dataset Loading & Validation

Auto-discover the CASIA dataset under `/kaggle/input/`, validate image-mask pairs, and perform stratified splitting.

In [ ]:
# ── Dataset Discovery ─────────────────────────────────────────────────────────
KAGGLE_INPUT = '/kaggle/input'

DATASET_ROOT = None
IMAGE_DIR = None
MASK_DIR = None

for root, dirs, files in os.walk(KAGGLE_INPUT):
    dirs_lower = [d.lower() for d in dirs]
    if 'image' in dirs_lower and 'mask' in dirs_lower:
        DATASET_ROOT = root
        IMAGE_DIR = os.path.join(root, dirs[dirs_lower.index('image')])
        MASK_DIR = os.path.join(root, dirs[dirs_lower.index('mask')])
        break

if DATASET_ROOT is None:
    raise FileNotFoundError(
        'Could not find dataset with IMAGE/ and MASK/ directories under /kaggle/input/. '
        'Ensure the CASIA Splicing Detection + Localization dataset is attached.'
    )

print(f'Dataset root: {DATASET_ROOT}')
print(f'IMAGE dir:    {IMAGE_DIR}')
print(f'MASK dir:     {MASK_DIR}')

for sub in ['Au', 'Tp']:
    img_sub = os.path.join(IMAGE_DIR, sub)
    mask_sub = os.path.join(MASK_DIR, sub)
    if os.path.isdir(img_sub):
        print(f'  {sub}: {len(os.listdir(img_sub))} images, '
              f'{len(os.listdir(mask_sub)) if os.path.isdir(mask_sub) else 0} masks')

In [ ]:
# ── Build Validated Pairs ──────────────────────────────────────────────────────

def is_valid_image(filepath):
    try:
        img = Image.open(filepath)
        img.verify()
        return True
    except Exception:
        return False

def detect_forgery_type(filename):
    fn = filename.lower()
    if '_cm' in fn or 'copy' in fn:
        return 'copy-move'
    elif '_sp' in fn or 'splic' in fn:
        return 'splicing'
    return 'unknown'

valid_exts = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}
label_folders = {'Au': {'class_name': 'authentic', 'label': 0},
                 'Tp': {'class_name': 'tampered',  'label': 1}}

pairs = []
excluded = []

for sub_name, info in label_folders.items():
    img_subdir = os.path.join(IMAGE_DIR, sub_name)
    mask_subdir = os.path.join(MASK_DIR, sub_name)

    if not os.path.isdir(img_subdir):
        print(f'Warning: {img_subdir} not found, skipping.')
        continue

    for fname in sorted(os.listdir(img_subdir)):
        ext = os.path.splitext(fname)[1].lower()
        if ext not in valid_exts:
            continue
        img_path = os.path.join(img_subdir, fname)
        mask_path = os.path.join(mask_subdir, fname) if os.path.isdir(mask_subdir) else None

        if mask_path and not os.path.exists(mask_path):
            # Try common mask name variants
            stem = os.path.splitext(fname)[0]
            for mext in ['.png', '.jpg', '.tif', '.bmp']:
                alt = os.path.join(mask_subdir, stem + mext)
                if os.path.exists(alt):
                    mask_path = alt
                    break
            else:
                mask_path = None

        entry = {
            'image_path': img_path,
            'mask_path': mask_path,
            'label': float(info['label']),
            'class_name': info['class_name'],
            'forgery_type': detect_forgery_type(fname) if info['label'] == 1 else 'authentic',
        }

        pairs.append(entry)

tampered_count = sum(1 for p in pairs if p['label'] == 1.0)
authentic_count = sum(1 for p in pairs if p['label'] == 0.0)

print('=' * 50)
print('DATASET VALIDATION SUMMARY')
print('=' * 50)
print(f'Total valid pairs:   {len(pairs)}')
print(f'  Tampered images:   {tampered_count}')
print(f'  Authentic images:  {authentic_count}')
print(f'  Tampered ratio:    {tampered_count / max(len(pairs), 1):.2%}')
forgery_types = Counter(p['forgery_type'] for p in pairs)
print(f'  Forgery types:     {dict(forgery_types)}')
print('=' * 50)

In [ ]:
# ── Stratified Split: 70/15/15 ────────────────────────────────────────────────

labels_for_split = [p['label'] for p in pairs]

train_pairs, temp_pairs = train_test_split(
    pairs, test_size=0.30, random_state=SEED, stratify=labels_for_split
)
temp_labels = [p['label'] for p in temp_pairs]
val_pairs, test_pairs = train_test_split(
    temp_pairs, test_size=0.5, random_state=SEED, stratify=temp_labels
)

print(f'Train: {len(train_pairs)}')
print(f'Val:   {len(val_pairs)}')
print(f'Test:  {len(test_pairs)}')

for name, split in [('Train', train_pairs), ('Val', val_pairs), ('Test', test_pairs)]:
    counts = Counter(p['class_name'] for p in split)
    print(f'  {name}: {dict(counts)}')

# Save split manifest
manifest = {
    'seed': SEED,
    'total_pairs': len(pairs),
    'train_count': len(train_pairs),
    'val_count': len(val_pairs),
    'test_count': len(test_pairs),
}
manifest_path = os.path.join(RESULTS_DIR, 'split_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'Split manifest saved to: {manifest_path}')

## 4. Preprocessing & Augmentation

**vK.4 augmentation pipeline** (adopted from v8):
- `ColorJitter` — photometric regularization
- `ImageCompression` — forces model to handle JPEG artifacts
- `GaussNoise` — noise robustness
- `GaussianBlur` — blur invariance

All augmentations are CONFIG-controlled.

In [ ]:
# ── Augmentation Pipeline ──────────────────────────────────────────────────────

def build_train_transform(config):
    transforms = [
        A.Resize(config['image_size'], config['image_size']),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.1, rotate_limit=10,
                           border_mode=cv2.BORDER_REFLECT_101, p=0.5),
    ]

    if config.get('aug_color_jitter', False):
        transforms.append(A.ColorJitter(brightness=0.2, contrast=0.2,
                                         saturation=0.2, hue=0.1, p=0.5))
    if config.get('aug_compression', False):
        transforms.append(A.ImageCompression(quality_lower=50, quality_upper=95, p=0.3))
    if config.get('aug_gauss_noise', False):
        transforms.append(A.GaussNoise(var_limit=(10, 50), p=0.3))
    if config.get('aug_gauss_blur', False):
        transforms.append(A.GaussianBlur(blur_limit=(3, 5), p=0.2))

    transforms.extend([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    return A.Compose(transforms)

train_transform = build_train_transform(CONFIG)
val_transform = A.Compose([
    A.Resize(CONFIG['image_size'], CONFIG['image_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print('Train augmentations:')
for t in train_transform.transforms:
    print(f'  {t.__class__.__name__}')

In [ ]:
# ── Dataset Class ─────────────────────────────────────────────────────────────

class TamperingDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        entry = self.pairs[idx]

        image = cv2.imread(entry['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if entry['mask_path'] is not None:
            mask = cv2.imread(entry['mask_path'], cv2.IMREAD_GRAYSCALE)
            mask = (mask > 0).astype(np.uint8)
        else:
            h, w = image.shape[:2]
            mask = np.zeros((h, w), dtype=np.uint8)

        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        if isinstance(mask, np.ndarray):
            mask = torch.from_numpy(mask)
        mask = mask.unsqueeze(0).float()

        label = torch.tensor(int(entry['label']), dtype=torch.long)
        return image, mask, label

In [ ]:
# ── DataLoaders ───────────────────────────────────────────────────────────────

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

train_dataset = TamperingDataset(train_pairs, transform=train_transform)
val_dataset = TamperingDataset(val_pairs, transform=val_transform)
test_dataset = TamperingDataset(test_pairs, transform=val_transform)

_nw = CONFIG['num_workers']
loader_kwargs = dict(
    num_workers=_nw,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=_nw > 0,
)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                          shuffle=True, drop_last=True,
                          worker_init_fn=seed_worker, generator=g, **loader_kwargs)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                        shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'],
                         shuffle=False, drop_last=False, **loader_kwargs)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')
print(f'num_workers={_nw}, pin_memory={loader_kwargs["pin_memory"]}')

In [ ]:
# ── Sanity Check: Visualize One Training Batch ────────────────────────────────

images, masks, labels = next(iter(train_loader))
print(f'Image batch: {images.shape}  Mask batch: {masks.shape}  Labels: {labels[:8]}')

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i in range(min(4, images.size(0))):
    img = images[i].permute(1, 2, 0).numpy() * std + mean
    img = np.clip(img, 0, 1)
    msk = masks[i].squeeze().numpy()

    axes[0, i].imshow(img)
    axes[0, i].set_title(f'Image (label={labels[i].item()})')
    axes[0, i].axis('off')

    axes[1, i].imshow(msk, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title('Mask')
    axes[1, i].axis('off')

plt.suptitle('Training Batch Sample', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Model Architecture

**Custom U-Net with Classifier Head** (preserved from vK.2):
- `DoubleConv`, `Down`, `Up` blocks form the encoder-decoder backbone
- Segmentation head outputs a 1-channel tampering mask
- Classification head on the bottleneck predicts authentic vs tampered

In [ ]:
# ── Model Architecture (preserved from vK.2) ─────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x):
        return self.conv(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                         diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNetWithClassifier(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, n_labels=2):
        super().__init__()
        # Encoder
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)
        # Decoder
        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)
        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)
        # Classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, n_labels),
        )

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)  # bottleneck
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        seg_logits = self.outc(x)
        cls_logits = self.classifier(x5)
        return cls_logits, seg_logits

model = UNetWithClassifier(
    n_channels=CONFIG['n_channels'],
    n_classes=CONFIG['n_classes'],
    n_labels=CONFIG['n_labels'],
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Shape check
with torch.no_grad():
    dummy = torch.randn(1, 3, CONFIG['image_size'], CONFIG['image_size']).to(device)
    cls_out, seg_out = model(dummy)
    print(f'Shape check: cls={cls_out.shape}, seg={seg_out.shape}')

## 6. Loss, Optimizer & Scheduler

**vK.4 improvements:**
1. **BCE `pos_weight`** — computed from training masks to fix class imbalance
2. **Per-sample Dice** — each image contributes equally to Dice gradient
3. **FocalLoss** for classification — handles class imbalance
4. **ReduceLROnPlateau** — reduces LR when val metric stalls

In [ ]:
# ── Compute pos_weight from Training Masks ────────────────────────────────────

pos_weight = None

if CONFIG['use_pos_weight']:
    print('Computing pos_weight from training masks...')
    total_fg, total_bg = 0, 0
    for pair in tqdm(train_pairs, desc='Scanning masks'):
        if pair['mask_path'] is not None:
            mask = cv2.imread(pair['mask_path'], cv2.IMREAD_GRAYSCALE)
            if mask is not None:
                fg = (mask > 0).sum()
                total_fg += fg
                total_bg += mask.size - fg
        else:
            total_bg += CONFIG['image_size'] ** 2

    pw_value = total_bg / max(total_fg, 1)
    pos_weight = torch.tensor([pw_value], dtype=torch.float32).to(device)
    print(f'pos_weight: {pw_value:.2f} (bg={total_bg:,}, fg={total_fg:,})')
    print(f'Foreground fraction: {total_fg / (total_fg + total_bg):.4%}')
else:
    print('pos_weight disabled.')

In [ ]:
# ── Loss Functions ────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, labels):
        ce = F.cross_entropy(logits, labels, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


class BCEDiceLoss(nn.Module):
    def __init__(self, pos_weight=None, smooth=1.0, per_sample_dice=True):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.smooth = smooth
        self.per_sample_dice = per_sample_dice

    def _dice_loss_per_sample(self, logits, targets):
        probs = torch.sigmoid(logits)
        B = probs.shape[0]
        p_flat = probs.view(B, -1)
        t_flat = targets.view(B, -1)
        inter = (p_flat * t_flat).sum(dim=1)
        union = p_flat.sum(dim=1) + t_flat.sum(dim=1)
        dice = (2.0 * inter + self.smooth) / (union + self.smooth)
        return 1.0 - dice.mean()

    def _dice_loss_batch(self, logits, targets):
        probs = torch.sigmoid(logits)
        p_flat = probs.view(-1)
        t_flat = targets.view(-1)
        inter = (p_flat * t_flat).sum()
        union = p_flat.sum() + t_flat.sum()
        return 1.0 - (2.0 * inter + self.smooth) / (union + self.smooth)

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        if self.per_sample_dice:
            dice = self._dice_loss_per_sample(logits, targets)
        else:
            dice = self._dice_loss_batch(logits, targets)
        return 0.5 * bce_loss + 0.5 * dice


# Classification loss
class_weights = compute_class_weight('balanced', classes=np.array([0, 1]),
                                      y=[int(p['label']) for p in train_pairs])
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f'Class weights: {class_weights}')

criterion_cls = FocalLoss(alpha=class_weights)
criterion_seg = BCEDiceLoss(pos_weight=pos_weight,
                             per_sample_dice=CONFIG['dice_per_sample'])

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'],
                              weight_decay=CONFIG['weight_decay'])

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=CONFIG['scheduler_factor'],
    patience=CONFIG['scheduler_patience'], min_lr=CONFIG['scheduler_min_lr'],
)

# AMP scaler
scaler = GradScaler('cuda', enabled=CONFIG['use_amp'])

print(f'Optimizer: Adam(lr={CONFIG["lr"]}, wd={CONFIG["weight_decay"]})')
print(f'Scheduler: ReduceLROnPlateau(patience={CONFIG["scheduler_patience"]})')
print(f'AMP enabled: {CONFIG["use_amp"]}')

In [ ]:
# ── Evaluation Metrics ────────────────────────────────────────────────────────

def compute_pixel_f1(pred, gt, eps=1e-8):
    pred, gt = pred.flatten(), gt.flatten()
    if gt.sum() == 0 and pred.sum() == 0:
        return 1.0
    if gt.sum() == 0 and pred.sum() > 0:
        return 0.0
    tp = (pred * gt).sum()
    fp = (pred * (1 - gt)).sum()
    fn = ((1 - pred) * gt).sum()
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    return (2 * precision * recall / (precision + recall + eps)).item()

def compute_iou(pred, gt, eps=1e-8):
    pred, gt = pred.flatten(), gt.flatten()
    if gt.sum() == 0 and pred.sum() == 0:
        return 1.0
    inter = (pred * gt).sum()
    union = pred.sum() + gt.sum() - inter
    return (inter / (union + eps)).item()

def dice_coef_batch(pred_logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(pred_logits)
    pred_bin = (probs > threshold).float()
    inter = (pred_bin * targets).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
    return ((2.0 * inter + eps) / (union + eps)).mean().item()

def iou_coef_batch(pred_logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(pred_logits)
    pred_bin = (probs > threshold).float()
    inter = (pred_bin * targets).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) - inter
    return ((inter + eps) / (union + eps)).mean().item()

def f1_coef_batch(pred_logits, targets, threshold=0.5, eps=1e-7):
    probs = torch.sigmoid(pred_logits)
    pred_bin = (probs > threshold).float()
    tp = (pred_bin * targets).sum(dim=(1,2,3))
    fp = (pred_bin * (1.0 - targets)).sum(dim=(1,2,3))
    fn = ((1.0 - pred_bin) * targets).sum(dim=(1,2,3))
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    return (2.0 * precision * recall / (precision + recall + eps)).mean().item()

print('Metric functions defined.')

## 7. Training Pipeline

**vK.4 training improvements:**
- Mixed-precision (AMP) with GradScaler
- Gradient accumulation (effective batch size = batch_size × accumulation_steps)
- Gradient clipping at `max_grad_norm=1.0`
- Early stopping on validation F1 (patience=10)
- Per-epoch LR logging
- `tqdm` progress bars

In [ ]:
# ── Training & Validation Functions ───────────────────────────────────────────

ALPHA = CONFIG['cls_loss_weight']
BETA = CONFIG['seg_loss_weight']

def train_one_epoch(model, loader, criterion_cls, criterion_seg, optimizer, scaler,
                    device, config):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    accum_steps = config['accumulation_steps']
    use_amp = config['use_amp']

    optimizer.zero_grad(set_to_none=True)
    pbar = tqdm(enumerate(loader), total=len(loader), leave=False, desc='Training')

    for batch_idx, (images, masks, labels) in pbar:
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)

        with autocast('cuda', enabled=use_amp):
            cls_logits, seg_logits = model(images)
            loss_cls = criterion_cls(cls_logits, labels)
            loss_seg = criterion_seg(seg_logits, masks)
            loss = (ALPHA * loss_cls + BETA * loss_seg) / accum_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accum_steps == 0 or (batch_idx + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * accum_steps * images.size(0)
        preds = torch.argmax(cls_logits, dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

        pbar.set_postfix({'loss': running_loss / total, 'acc': correct / total})

    return running_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion_cls, criterion_seg, device, config):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    dice_sum, iou_sum, f1_sum = 0.0, 0.0, 0.0
    num_batches = 0
    use_amp = config['use_amp']

    for images, masks, labels in loader:
        images = images.to(device)
        masks = masks.to(device)
        labels = labels.to(device)

        with autocast('cuda', enabled=use_amp):
            cls_logits, seg_logits = model(images)
            loss_cls = criterion_cls(cls_logits, labels)
            loss_seg = criterion_seg(seg_logits, masks)
            loss = ALPHA * loss_cls + BETA * loss_seg

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(cls_logits, dim=1)
        correct += (preds == labels).sum().item()
        total += images.size(0)

        dice_sum += dice_coef_batch(seg_logits, masks)
        iou_sum += iou_coef_batch(seg_logits, masks)
        f1_sum += f1_coef_batch(seg_logits, masks)
        num_batches += 1

    metrics = {
        'loss': running_loss / total,
        'acc': correct / total,
        'dice': dice_sum / max(num_batches, 1),
        'iou': iou_sum / max(num_batches, 1),
        'f1': f1_sum / max(num_batches, 1),
    }
    return metrics

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [],
    'val_dice': [], 'val_iou': [], 'val_f1': [],
    'lr': [],
}

best_val_f1 = 0.0
best_epoch = 0
patience_counter = 0

NUM_EPOCHS = CONFIG['max_epochs']
best_model_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')

print(f'Starting training for {NUM_EPOCHS} epochs...')
print(f'Early stopping patience: {CONFIG["patience"]}')
print('=' * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    current_lr = optimizer.param_groups[0]['lr']
    print(f'\nEpoch {epoch:02d}/{NUM_EPOCHS} | LR: {current_lr:.2e}')

    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion_cls, criterion_seg,
        optimizer, scaler, device, CONFIG
    )

    val_metrics = validate(model, val_loader, criterion_cls, criterion_seg, device, CONFIG)

    # Update scheduler
    scheduler.step(val_metrics['f1'])

    # Log history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_metrics['loss'])
    history['val_acc'].append(val_metrics['acc'])
    history['val_dice'].append(val_metrics['dice'])
    history['val_iou'].append(val_metrics['iou'])
    history['val_f1'].append(val_metrics['f1'])
    history['lr'].append(current_lr)

    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'  Val   Loss: {val_metrics["loss"]:.4f} | Val Acc: {val_metrics["acc"]:.4f} | '
          f'Dice: {val_metrics["dice"]:.4f} | IoU: {val_metrics["iou"]:.4f} | F1: {val_metrics["f1"]:.4f}')

    # W&B logging
    if WANDB_ACTIVE:
        wandb.log({
            'epoch': epoch, 'lr': current_lr,
            'train/loss': train_loss, 'train/accuracy': train_acc,
            'val/loss': val_metrics['loss'], 'val/accuracy': val_metrics['acc'],
            'val/dice': val_metrics['dice'], 'val/iou': val_metrics['iou'],
            'val/f1': val_metrics['f1'],
        })

    # Checkpointing
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        best_epoch = epoch
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_f1': best_val_f1,
            'config': CONFIG,
        }, best_model_path)
        print(f'  ==> Saved best model (F1: {best_val_f1:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= CONFIG['patience']:
            print(f'  Early stopping triggered after {CONFIG["patience"]} epochs without improvement.')
            break

    torch.cuda.empty_cache()

print('\n' + '=' * 60)
print(f'Training finished. Best Val F1: {best_val_f1:.4f} at epoch {best_epoch}')
print('=' * 60)

## 8. Training Curves

In [ ]:
# ── Training Curves ────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0, 0].plot(epochs_range, history['train_loss'], label='Train')
axes[0, 0].plot(epochs_range, history['val_loss'], label='Val')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(epochs_range, history['val_f1'], color='green')
axes[0, 1].axvline(x=best_epoch, color='red', linestyle='--',
                    label=f'Best (epoch {best_epoch})')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('F1')
axes[0, 1].set_title('Validation F1')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(epochs_range, history['val_dice'], label='Dice')
axes[1, 0].plot(epochs_range, history['val_iou'], label='IoU')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Validation Segmentation Metrics')
axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(epochs_range, history['lr'], color='purple')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Learning Rate')
axes[1, 1].set_title('Learning Rate Schedule')
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Training Summary', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved.')

## 9. Evaluation

**vK.4 evaluation improvements:**
1. Load best checkpoint
2. Threshold sweep on validation set (0.05 to 0.80)
3. Full test evaluation with mask-size stratification
4. Tampered-only metrics reported separately

In [ ]:
# ── Load Best Model ────────────────────────────────────────────────────────────

ckpt = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Loaded best model from epoch {ckpt["epoch"]} (F1={ckpt["best_f1"]:.4f})')

In [ ]:
# ── Threshold Sweep ────────────────────────────────────────────────────────────

@torch.no_grad()
def find_best_threshold(model, loader, device, config, thresholds=None):
    model.eval()
    if thresholds is None:
        thresholds = np.arange(0.05, 0.80, 0.05)

    all_probs, all_masks = [], []
    for images, masks, labels in tqdm(loader, desc='Collecting val predictions'):
        images = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            cls_logits, seg_logits = model(images)
        probs = torch.sigmoid(seg_logits).cpu()
        all_probs.append(probs)
        all_masks.append(masks)

    all_probs = torch.cat(all_probs, dim=0)
    all_masks = torch.cat(all_masks, dim=0)

    results = []
    for t in thresholds:
        preds = (all_probs > t).float()
        f1_scores = []
        for i in range(preds.shape[0]):
            f1_scores.append(compute_pixel_f1(preds[i], all_masks[i]))
        mean_f1 = np.mean(f1_scores)
        results.append((t, mean_f1))
        print(f'  Threshold {t:.2f}: F1={mean_f1:.4f}')

    best_t, best_f1_val = max(results, key=lambda x: x[1])
    return best_t, results

best_threshold, threshold_results = find_best_threshold(model, val_loader, device, CONFIG)
print(f'\nBest threshold: {best_threshold:.3f}')

In [ ]:
# ── Full Test Evaluation ───────────────────────────────────────────────────────

@torch.no_grad()
def evaluate_test(model, loader, test_pairs, device, config, threshold):
    model.eval()
    all_f1, all_iou = [], []
    tampered_f1, tampered_iou = [], []
    image_preds, image_labels = [], []

    size_buckets = {
        'tiny (<2%)': {'range': (0, 0.02), 'f1': []},
        'small (2-5%)': {'range': (0.02, 0.05), 'f1': []},
        'medium (5-15%)': {'range': (0.05, 0.15), 'f1': []},
        'large (>15%)': {'range': (0.15, 1.0), 'f1': []},
    }

    idx = 0
    for images, masks, labels in tqdm(loader, desc='Test evaluation'):
        images = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            cls_logits, seg_logits = model(images)
        probs = torch.sigmoid(seg_logits).cpu()
        cls_preds = torch.argmax(cls_logits, dim=1).cpu()

        for i in range(images.size(0)):
            pred_mask = (probs[i] > threshold).float()
            f1 = compute_pixel_f1(pred_mask, masks[i])
            iou = compute_iou(pred_mask, masks[i])
            all_f1.append(f1)
            all_iou.append(iou)

            image_preds.append(cls_preds[i].item())
            image_labels.append(labels[i].item())

            if idx < len(test_pairs) and test_pairs[idx]['label'] == 1.0:
                tampered_f1.append(f1)
                tampered_iou.append(iou)

                gt_area = masks[i].sum().item() / masks[i].numel()
                for bname, bdata in size_buckets.items():
                    lo, hi = bdata['range']
                    if lo <= gt_area < hi:
                        bdata['f1'].append(f1)
                        break

            idx += 1

    results = {
        'pixel_f1_mean': np.mean(all_f1),
        'pixel_iou_mean': np.mean(all_iou),
        'tampered_f1_mean': np.mean(tampered_f1) if tampered_f1 else 0.0,
        'tampered_iou_mean': np.mean(tampered_iou) if tampered_iou else 0.0,
        'image_accuracy': np.mean([p == l for p, l in zip(image_preds, image_labels)]),
        'threshold': threshold,
    }

    print('\n' + '=' * 60)
    print('TEST RESULTS')
    print('=' * 60)
    print(f'  Threshold:            {threshold:.3f}')
    print(f'  Image Accuracy:       {results["image_accuracy"]:.4f}')
    print(f'  Pixel F1 (all):       {results["pixel_f1_mean"]:.4f}')
    print(f'  Pixel IoU (all):      {results["pixel_iou_mean"]:.4f}')
    print(f'  Pixel F1 (tampered):  {results["tampered_f1_mean"]:.4f}')

    print('\n  Mask-Size Stratification:')
    for bname, bdata in size_buckets.items():
        if bdata['f1']:
            print(f'    {bname}: F1={np.mean(bdata["f1"]):.4f} (n={len(bdata["f1"])})')
        else:
            print(f'    {bname}: no samples')

    return results

test_results = evaluate_test(model, test_loader, test_pairs, device, CONFIG, best_threshold)

## 10. Visualization

Prediction grids: Original | GT Mask | Predicted Mask | Overlay

In [ ]:
# ── Denormalize Helper ─────────────────────────────────────────────────────────
def denormalize(img_tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = img_tensor.cpu() * std + mean
    return torch.clamp(img, 0, 1)

# ── Collect predictions for visualization ─────────────────────────────────────
@torch.no_grad()
def collect_predictions(model, loader, test_pairs, device, config, threshold):
    model.eval()
    predictions = []
    idx = 0
    for images, masks, labels in loader:
        images_dev = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            cls_logits, seg_logits = model(images_dev)
        probs = torch.sigmoid(seg_logits).cpu()
        cls_preds = torch.argmax(cls_logits, dim=1).cpu()

        for i in range(images.size(0)):
            pred_mask = (probs[i] > threshold).float()
            f1 = compute_pixel_f1(pred_mask, masks[i])
            predictions.append({
                'image': images[i],
                'gt_mask': masks[i],
                'pred_mask': pred_mask,
                'label': labels[i].item(),
                'pred_label': cls_preds[i].item(),
                'pixel_f1': f1,
                'forgery_type': test_pairs[idx]['forgery_type'] if idx < len(test_pairs) else 'unknown',
                'gt_mask_area': masks[i].sum().item() / masks[i].numel(),
            })
            idx += 1
    return predictions

predictions = collect_predictions(model, test_loader, test_pairs, device, CONFIG, best_threshold)
print(f'Collected {len(predictions)} predictions.')

In [ ]:
# ── Prediction Grid ───────────────────────────────────────────────────────────

tampered_preds = [p for p in predictions if p['label'] == 1]
authentic_preds = [p for p in predictions if p['label'] == 0]
tampered_sorted = sorted(tampered_preds, key=lambda p: p['pixel_f1'])

samples = []
if len(tampered_sorted) >= 2:
    samples.extend(tampered_sorted[-2:])   # Best
mid = len(tampered_sorted) // 2
if len(tampered_sorted) >= 4:
    samples.extend(tampered_sorted[mid-1:mid+1])  # Median
if len(tampered_sorted) >= 2:
    samples.extend(tampered_sorted[:2])    # Worst
if len(authentic_preds) >= 2:
    samples.extend(authentic_preds[:2])    # Authentic

n_rows = len(samples)
if n_rows > 0:
    fig, axes = plt.subplots(n_rows, 4, figsize=(20, 5 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    col_titles = ['Original', 'GT Mask', 'Predicted Mask', 'Overlay']

    for row, s in enumerate(samples):
        img = denormalize(s['image']).permute(1, 2, 0).numpy()
        gt = s['gt_mask'].squeeze().numpy()
        pred = s['pred_mask'].squeeze().numpy()

        overlay = img.copy()
        overlay[pred > 0.5] = [1, 0, 0]
        blended = 0.6 * img + 0.4 * overlay

        lbl = 'Authentic' if s['label'] == 0 else 'Tampered'
        pred_lbl = 'Authentic' if s['pred_label'] == 0 else 'Tampered'

        for col, data in enumerate([img, gt, pred, blended]):
            cmap = 'gray' if col in [1, 2] else None
            axes[row, col].imshow(data, cmap=cmap, vmin=0, vmax=1)
            if row == 0:
                axes[row, col].set_title(col_titles[col], fontsize=12)
            axes[row, col].axis('off')

        axes[row, 0].set_ylabel(f'{lbl}\nPred: {pred_lbl}\nF1: {s["pixel_f1"]:.3f}',
                                 fontsize=10, rotation=0, labelpad=80, va='center')

    plt.suptitle('Prediction Grid (Best / Median / Worst / Authentic)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'prediction_grid.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No samples available for prediction grid.')

In [ ]:
# ── F1 vs Threshold Plot ──────────────────────────────────────────────────────

thresh_vals = [r[0] for r in threshold_results]
f1_vals = [r[1] for r in threshold_results]

plt.figure(figsize=(8, 5))
plt.plot(thresh_vals, f1_vals, 'b-', linewidth=2)
plt.axvline(x=best_threshold, color='red', linestyle='--',
            label=f'Best: {best_threshold:.3f}')
plt.xlabel('Threshold'); plt.ylabel('Mean Pixel-F1')
plt.title('F1 vs. Threshold (Validation Set)')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'f1_vs_threshold.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Explainable AI — Grad-CAM

Spatial attention heatmaps from the deepest encoder layer.

In [ ]:
# ── Grad-CAM ──────────────────────────────────────────────────────────────────

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        self._handles = []
        self._handles.append(target_layer.register_forward_hook(self._save_act))
        self._handles.append(target_layer.register_full_backward_hook(self._save_grad))

    def _save_act(self, module, inp, out):
        self.activations = out.detach()
    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor):
        self.model.eval()
        cls_logits, seg_logits = self.model(input_tensor)
        target = seg_logits.sum()
        self.model.zero_grad()
        target.backward()

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam.cpu().numpy()

    def remove_hooks(self):
        for h in self._handles:
            h.remove()

# Visualize Grad-CAM on tampered samples
grad_cam = GradCAM(model, model.down4.conv.block)

cam_samples = sorted([p for p in predictions if p['label'] == 1],
                      key=lambda p: p['pixel_f1'], reverse=True)[:4]

if cam_samples:
    fig, axes = plt.subplots(len(cam_samples), 4, figsize=(20, 5 * len(cam_samples)))
    if len(cam_samples) == 1:
        axes = axes[np.newaxis, :]

    for row, s in enumerate(cam_samples):
        img_tensor = s['image'].unsqueeze(0).to(device).requires_grad_(True)
        cam = grad_cam.generate(img_tensor)
        img_np = denormalize(s['image']).permute(1, 2, 0).numpy()
        gt_mask = s['gt_mask'].squeeze().numpy()
        pred = s['pred_mask'].squeeze().numpy()

        axes[row, 0].imshow(img_np); axes[row, 0].set_title('Image'); axes[row, 0].axis('off')
        axes[row, 1].imshow(gt_mask, cmap='gray'); axes[row, 1].set_title('GT Mask'); axes[row, 1].axis('off')
        axes[row, 2].imshow(pred, cmap='gray'); axes[row, 2].set_title(f'Pred (F1={s["pixel_f1"]:.3f})'); axes[row, 2].axis('off')
        axes[row, 3].imshow(img_np); axes[row, 3].imshow(cam, cmap='jet', alpha=0.5)
        axes[row, 3].set_title('Grad-CAM'); axes[row, 3].axis('off')

    plt.suptitle('Grad-CAM Analysis', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, 'gradcam_analysis.png'), dpi=150, bbox_inches='tight')
    plt.show()

grad_cam.remove_hooks()

## 12. Robustness Testing

Evaluate under degradation: JPEG compression, Gaussian noise, blur, resize.

In [ ]:
# ── Robustness Evaluation ──────────────────────────────────────────────────────

NORMALIZE = A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
SZ = CONFIG['image_size']

robustness_transforms = {
    'clean': A.Compose([A.Resize(SZ, SZ), NORMALIZE, ToTensorV2()]),
    'jpeg_qf70': A.Compose([A.Resize(SZ, SZ), A.ImageCompression(quality_lower=70, quality_upper=70, p=1.0), NORMALIZE, ToTensorV2()]),
    'jpeg_qf50': A.Compose([A.Resize(SZ, SZ), A.ImageCompression(quality_lower=50, quality_upper=50, p=1.0), NORMALIZE, ToTensorV2()]),
    'gauss_noise': A.Compose([A.Resize(SZ, SZ), A.GaussNoise(var_limit=(20, 60), p=1.0), NORMALIZE, ToTensorV2()]),
    'gauss_blur': A.Compose([A.Resize(SZ, SZ), A.GaussianBlur(blur_limit=(5, 7), p=1.0), NORMALIZE, ToTensorV2()]),
    'resize_half': A.Compose([A.Resize(SZ // 2, SZ // 2), A.Resize(SZ, SZ), NORMALIZE, ToTensorV2()]),
}

@torch.no_grad()
def run_robustness_eval(model, loader, device, config, threshold):
    model.eval()
    f1_scores = []
    for images, masks, labels in loader:
        images = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            cls_logits, seg_logits = model(images)
        probs = torch.sigmoid(seg_logits).cpu()
        preds = (probs > threshold).float()
        for i in range(images.size(0)):
            f1_scores.append(compute_pixel_f1(preds[i], masks[i]))
    return f1_scores

robustness_results = {}
for name, tfm in robustness_transforms.items():
    ds = TamperingDataset(test_pairs, transform=tfm)
    dl = DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
    f1s = run_robustness_eval(model, dl, device, CONFIG, best_threshold)
    robustness_results[name] = {'f1_mean': np.mean(f1s), 'f1_std': np.std(f1s)}
    print(f'{name:20s}: F1 = {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')

# Chart
names = list(robustness_results.keys())
means = [robustness_results[n]['f1_mean'] for n in names]
stds = [robustness_results[n]['f1_std'] for n in names]

plt.figure(figsize=(12, 6))
bars = plt.bar(range(len(names)), means, yerr=stds, capsize=4, color='steelblue', alpha=0.8)
if 'clean' in names:
    bars[names.index('clean')].set_color('green')
plt.xticks(range(len(names)), names, rotation=45, ha='right')
plt.ylabel('Pixel-F1'); plt.title('Robustness Testing')
plt.grid(True, axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'robustness_chart.png'), dpi=150, bbox_inches='tight')
plt.show()

## 13. Shortcut Learning Checks

Verify the model isn't relying on dataset artifacts.

In [ ]:
# ── Mask Randomization Test ────────────────────────────────────────────────────

@torch.no_grad()
def mask_randomization_test(model, loader, device, config, threshold, n_batches=10):
    model.eval()
    f1_scores = []
    batch_count = 0
    for images, masks, labels in loader:
        if batch_count >= n_batches:
            break
        images = images.to(device)
        with autocast('cuda', enabled=config['use_amp']):
            cls_logits, seg_logits = model(images)
        probs = torch.sigmoid(seg_logits).cpu()
        preds = (probs > threshold).float()
        for i in range(images.size(0)):
            random_mask = torch.randint(0, 2, masks[i].shape).float()
            f1_scores.append(compute_pixel_f1(preds[i], random_mask))
        batch_count += 1

    mean_f1 = np.mean(f1_scores)
    print(f'Mask Randomization Test: F1 = {mean_f1:.4f} (should be ~0.0)')
    if mean_f1 > 0.1:
        print('  WARNING: F1 against random masks is suspiciously high!')
    else:
        print('  PASS: Model is not predicting dataset-correlated patterns.')
    return mean_f1

random_f1 = mask_randomization_test(model, test_loader, device, CONFIG, best_threshold)

## 14. Save Artifacts

In [ ]:
# ── Save Results Summary ──────────────────────────────────────────────────────

results_summary = {
    'version': 'vK.4',
    'config': CONFIG,
    'seed': SEED,
    'best_epoch': best_epoch,
    'best_val_f1': float(best_val_f1),
    'threshold': float(best_threshold),
    'test_results': test_results,
    'robustness_results': {
        name: {'f1_mean': float(d['f1_mean']), 'f1_std': float(d['f1_std'])}
        for name, d in robustness_results.items()
    },
    'random_mask_f1': float(random_f1),
}

summary_path = os.path.join(RESULTS_DIR, 'results_summary.json')
with open(summary_path, 'w') as f:
    json.dump(results_summary, f, indent=2, default=str)
print(f'Results saved to: {summary_path}')

# W&B final logging
if WANDB_ACTIVE:
    wandb.summary.update({
        'best/val_f1': best_val_f1,
        'best/epoch': best_epoch,
        'test/pixel_f1_all': test_results['pixel_f1_mean'],
        'test/pixel_f1_tampered': test_results['tampered_f1_mean'],
        'test/image_accuracy': test_results['image_accuracy'],
        'test/threshold': best_threshold,
    })
    # Upload model artifact
    artifact = wandb.Artifact('best-model-vk4', type='model')
    if os.path.exists(best_model_path):
        artifact.add_file(best_model_path)
        wandb.log_artifact(artifact)
    wandb.finish()
    print('W&B run finished.')

In [ ]:
# ── Artifact Inventory ────────────────────────────────────────────────────────

print()
print('=' * 60)
print('NOTEBOOK vK.4 COMPLETE — ARTIFACT INVENTORY')
print('=' * 60)

expected = {
    CHECKPOINT_DIR: ['best_model.pt'],
    RESULTS_DIR: ['split_manifest.json', 'results_summary.json'],
    PLOTS_DIR: ['training_curves.png', 'f1_vs_threshold.png',
                'prediction_grid.png', 'gradcam_analysis.png', 'robustness_chart.png'],
}

all_ok = True
for directory, files in expected.items():
    print(f'\n{directory}/')
    for fname in files:
        fpath = os.path.join(directory, fname)
        status = 'OK' if os.path.exists(fpath) else 'MISSING'
        if status == 'MISSING':
            all_ok = False
        size = os.path.getsize(fpath) / 1024 if os.path.exists(fpath) else 0
        print(f'  [{status}] {fname} ({size:.1f} KB)')

print('\n' + ('All artifacts present.' if all_ok else 'Some artifacts missing.'))

## Conclusion

This notebook presents a complete, Kaggle-optimized pipeline for tampered image detection and localization.

**Improvements over vK.2**
- Kaggle-native execution (no Colab/Drive shims)
- Centralized CONFIG, full reproducibility, AMP, gradient accumulation
- pos_weight BCE + per-sample Dice loss for better segmentation
- Expanded augmentation → improved robustness
- Threshold sweep + mask-size stratified evaluation
- Grad-CAM explainability + shortcut learning verification

**Model preserved from vK.2:** Custom `UNetWithClassifier` with DoubleConv encoder-decoder and classification head on bottleneck features.

**Artifacts produced:** best model checkpoint, training curves, prediction grids, Grad-CAM analysis, robustness chart, and JSON results summary.